In [1]:
import numpy as np
import pandas as pd
import ast
import nltk
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle


In [2]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')


In [3]:
movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800


In [4]:
credits.head(1)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [5]:
movies = movies.merge(credits, on='title')


In [6]:
movies.shape

(4809, 23)

In [7]:
credits.shape

(4803, 4)

In [8]:
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]


In [9]:
movies.dropna(inplace=True)


In [11]:
def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)


In [10]:
def convert_cast(obj):
    L = []
    counter = 0
    for i in ast.literal_eval(obj):
        if counter != 3:
            L.append(i['name'])
            counter += 1
        else:
            break
    return L

movies['cast'] = movies['cast'].apply(convert_cast)


In [12]:
def fetch_director(obj):
    L = []
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            L.append(i['name'])
            break
    return L

movies['crew'] = movies['crew'].apply(fetch_director)


In [13]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())


In [14]:
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])


In [16]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

new_df = movies[['movie_id','title','tags']].copy()

new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())


In [17]:
ps = PorterStemmer()

def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

new_df['tags'] = new_df['tags'].apply(stem)


In [18]:
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()


In [19]:
similarity = cosine_similarity(vectors)


In [20]:
def recommend(movie):
    movie_index = new_df[new_df['title'] == movie].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)), 
                         reverse=True, 
                         key=lambda x: x[1])[1:6]

    for i in movies_list:
        print(new_df.iloc[i[0]].title)


In [21]:
recommend('Avatar')


Aliens vs Predator: Requiem
Aliens
Falcon Rising
Independence Day
Titan A.E.


In [22]:
import pickle

pickle.dump(movies, open('movies.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))


In [23]:
new_df = pickle.load(open('movies.pkl','rb'))
similarity = pickle.load(open('similarity.pkl','rb'))


In [24]:
new_df['title'].sample(20)


1680                             United Passions
1642                               North Country
2245                              I'm Not There.
271                                   The Island
583                                     Big Fish
4741                           The Night Visitor
4580                                    Roadside
2886                                      Stoker
4608                                12 Angry Men
3603    Halloween 4: The Return of Michael Myers
3204                     Veronika Decides to Die
2836                                    How High
3804                       A Lonely Place to Die
1314                              The Ninth Gate
4381                                    Blue Car
985                                Into the Blue
2248                        Synecdoche, New York
1438                             Furry Vengeance
4478                                   Kill List
1904                                     The Box
Name: title, dtype: 

In [25]:
recommend("The Dark Knight Rises")


The Dark Knight
Batman Returns
Batman
Batman Forever
Batman Begins


In [26]:
new_df['title'].sample(20)

1481                        Exit Wounds
4261                          Checkmate
2036                       Delivery Man
273               Gone in Sixty Seconds
2621                          Post Grad
3168                         Set It Off
1349                              Aloha
4756                        The Dirties
1053                       Training Day
4450                               Elza
2427                               They
4769        Smiling Fish & Goat On Fire
2605                         Home Fries
1719                         Safe Haven
3254    On Her Majesty's Secret Service
3417                          Incendies
2734                Save the Last Dance
3826                     The Good Heart
876                Domestic Disturbance
3686             Let's Kill Ward's Wife
Name: title, dtype: object

In [27]:
recommend("Spider-Man 3")

Spider-Man 2
Spider-Man
The Amazing Spider-Man 2
The Amazing Spider-Man
Arachnophobia


In [28]:
recommend("Titanic")

The Notebook
Under the Same Moon
Ghost Ship
The Bounty
Pirates of the Caribbean: On Stranger Tides


In [29]:
import os
os.listdir()


['tmdb_5000_credits.csv',
 'app.py',
 'similarity.pkl',
 'tmdb_5000_movies.csv',
 'dummy.png',
 'dummy1.jpg',
 'movies.pkl',
 'movie recommender system.ipynb',
 '.ipynb_checkpoints',
 'dummy.jpg',
 'streamlit.ipynb']

In [30]:
import os
os.getcwd()


'/home/student/MACHINE_LEARNING_PROJECTS/MOVIE_RECOMMDER_SYSTEM'

In [31]:
movies.head()


,movie_id,title,overview,genres,keywords,cast,crew,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron],"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski],"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...","[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes],"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dccomics, crimefighter, terrorist, secretiden...","[ChristianBale, MichaelCaine, GaryOldman]",[ChristopherNolan],"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, ScienceFiction]","[basedonnovel, mars, medallion, spacetravel, p...","[TaylorKitsch, LynnCollins, SamanthaMorton]",[AndrewStanton],"[John, Carter, is, a, war-weary,, former, mili..."


In [32]:
movies.columns


Index(['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew',
       'tags'],
      dtype='object')